# VIA v0140B Windows Handle Validation — Evidence Review
此 notebook **只讀取 run-local evidence**，不開啟來源 CSV、不計算來源 hash、不套用 patch。先設定 `VIA_WINDOWS_HANDLE_RUN_DIR` 指向已完成的 run directory。

In [ ]:
import csv, json, os
from pathlib import Path

run_dir = Path(os.environ['VIA_WINDOWS_HANDLE_RUN_DIR']).resolve()
summary = json.loads((run_dir / 'VIA_v0140B_WindowsHandle_Summary.json').read_text(encoding='utf-8-sig'))
summary

In [ ]:
with (run_dir / 'VIA_v0140B_WindowsHandle_EngineMatrix.csv').open(encoding='utf-8-sig', newline='') as handle:
    engines = list(csv.DictReader(handle))
[(row['SEQUENCE_INDEX'], row['ENGINE_ID'], row['OUTCOME'], row['SOURCE_OPEN_ATTEMPTS'], row['SOURCE_OPEN_SUCCESSES'], row['SOURCE_BYTES_READ']) for row in engines]

In [ ]:
zero_fields = ['NETWORK_REQUESTS', 'SOURCE_HASHES', 'SOURCE_WRITES', 'SOURCE_COPIES', 'SOURCE_CONTENT_ARTIFACTS', 'CANONICAL_RUNTIME', 'REGISTRY_WRITES', 'EXISTING_MUTATION', 'PATCH_APPLICATIONS', 'CANDIDATE_PROMOTIONS']
violations = [(row['ENGINE_ID'], field, row[field]) for row in engines for field in zero_fields if int(row[field]) != 0]
assert len(engines) == 6
assert summary['cross_subsystem_rows'] == 18
assert not violations
{'gate': summary['final_gate'], 'attempts': summary['source_open_attempts'], 'successes': summary['source_open_successes'], 'bytes': summary['source_bytes_read'], 'hydra': summary['hydra_violations']}

## 判讀
PASS 僅表示至少一個 handle strategy 已安全取得有限位元組。若為 hydration-risk HOLD，請勿用此 notebook 或其他工具開啟來源內容；先人工檢視 OneDrive placeholder / recall 狀態。